In [1]:
from IPython.display import HTML

HTML(
    '<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">'
)

# <i class="fas fa-user-graduate" style="color:#18ab4b"></i> BYOC training: CatBoost credit model

Build a <i class="fas fa-file-code" style="color:#6c63ff"></i> **bring-your-own-container** (CatBoost), train it with the SageMaker SDK v3 `ModelTrainer` in local mode (`Mode.LOCAL_CONTAINER`, a real Docker training job on the `/opt/ml` contract), and store the <i class="fas fa-brain" style="color:#18ab4b"></i> model in oblako's <i class="fas fa-database" style="color:#18ab4b"></i> **S3Proxy**. Swap `Mode.LOCAL_CONTAINER` for the default `Mode.SAGEMAKER_TRAINING_JOB` and the same trainer runs as a managed job. Mirrors `sagemaker_endpoints/ml_models`.

Prereqs: `oblako up s3`, Docker, `pip install 'oblako[sagemaker,notebook]'`.

In [2]:
import logging

# the SageMaker SDK logs every step at INFO through a root rich handler, unless the
# root logger is configured first: keep its output (and botocore's) to warnings
logging.basicConfig(level=logging.WARNING)
logging.getLogger("sagemaker.config").setLevel(logging.WARNING)
# a local endpoint is polled on /ping while it starts; the retries aren't news
logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)

## <i class="fas fa-file-code" style="color:#6c63ff"></i> Build the container + <i class="fas fa-user-graduate" style="color:#18ab4b"></i> train

In [3]:
import os
import shutil
from pathlib import Path

import creditdata
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, InputData
from sagemaker.train.model_trainer import Mode

from oblako.engines.sagemaker import use_local_stubs
from oblako.services import SageMakerService

os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
use_local_stubs()  # no AWS account: skip the SDK's IAM role check

IMAGE = "oblako-credit-risk:latest"
HP = {
    "iterations": "150",
    "depth": "6",
    "learning-rate": "0.1",
    "factor": "20",
    "offset": "600",
}

sm = SageMakerService()
sm.build_image(path="container", tag=IMAGE)  # CatBoost BYOC (./container)

# ModelTrainer's local mode bind-mounts its working dir into the container, so keep
# it under the project: Docker Desktop shares /Users, not the /var/folders temp dir.
work = Path(".sm-local/01").resolve()
shutil.rmtree(work, ignore_errors=True)
(work / "train").mkdir(parents=True)
(work / "job").mkdir()
creditdata.write_training_csv(str(work / "train" / "train.csv"), n=600)

trainer = ModelTrainer(
    training_image=IMAGE,
    training_mode=Mode.LOCAL_CONTAINER,
    compute=Compute(instance_type="local", instance_count=1),
    role="arn:aws:iam::000000000000:role/dummy",
    hyperparameters=HP,
    local_container_root=str(work / "job"),
)
trainer.train(
    input_data_config=[InputData(channel_name="train", data_source=str(work / "train"))]
)
model_tar = work / "job" / "compressed_artifacts" / "model.tar.gz"
print("model:", model_tar.relative_to(Path.cwd()))

Step 1/7 : FROM python:3.11-slim
 ---> 1042b61448fe
Step 2/7 : RUN apt-get update && apt-get install -y --no-install-recommends libgomp1     && rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> 6f2747b28505
Step 3/7 : RUN pip install --no-cache-dir catboost numpy pandas joblib
 ---> Using cache
 ---> b722ab224512
Step 4/7 : COPY src/ /opt/program/
 ---> Using cache
 ---> e61966507729
Step 5/7 : WORKDIR /opt/program
 ---> Using cache
 ---> 260c052be1a8
Step 6/7 : ENV PYTHONUNBUFFERED=1
 ---> Using cache
 ---> 6d875257c67c
Step 7/7 : ENTRYPOINT ["python", "main.py"]
 ---> Using cache
 ---> e3ca3a20dd4e
Successfully built e3ca3a20dd4e
Successfully tagged oblako-credit-risk:latest


time="2026-09-30T18:57:12+02:00" level=warning msg="a network with name sagemaker-local exists but was not created for project \"job\".\nSet `external: true` to use an existing network"
 Container job-algo-1-1 Creating 
 Container job-algo-1-1 Created 
Attaching to algo-1-1
 Container job-algo-1-1 Starting 
 Container job-algo-1-1 Started 


algo-1-1  | trained CatBoost on 600 rows, 12 features (default rate 0.39)
algo-1-1 exited with code 0
 Compose Stopping Aborting on container exit...
 Container job-algo-1-1 Stopping 
 Container job-algo-1-1 Stopped 


model: .sm-local/01/job/compressed_artifacts/model.tar.gz


## <i class="fas fa-database" style="color:#18ab4b"></i> Store the trained model in oblako's S3

In [4]:
from oblako.services import S3ProxyService

s3 = S3ProxyService().get_client()
try:
    s3.create_bucket(Bucket="credit-risk-models")
except Exception:
    pass
s3.upload_file(str(model_tar), "credit-risk-models", "credit-risk/model.tar.gz")
print("stored: s3://credit-risk-models/credit-risk/model.tar.gz")
print(
    "bucket:",
    [
        o["Key"]
        for o in s3.list_objects_v2(Bucket="credit-risk-models").get("Contents", [])
    ],
)

stored: s3://credit-risk-models/credit-risk/model.tar.gz
bucket: ['credit-risk/model.tar.gz']
